In [1]:
import xml.etree.ElementTree as ET
import math
import re

def parse_path(d):
    # Tokenize commands and numbers
    tokens = re.findall(r'[MLHVZmlhvz]|-?\d+\.?\d*', d)
    lines = []
    current = (0,0)
    start = None
    i = 0
    while i < len(tokens):
        cmd = tokens[i]
        if cmd in ['M','m']:
            x = float(tokens[i+1]); y = float(tokens[i+2])
            current = (x,y)
            start = current
            i += 3
        elif cmd in ['L','l']:
            x = float(tokens[i+1]); y = float(tokens[i+2])
            new_point = (x,y)
            lines.append((current,new_point))
            current = new_point
            i += 3
        elif cmd in ['H','h']:
            x = float(tokens[i+1])
            new_point = (x,current[1])
            lines.append((current,new_point))
            current = new_point
            i += 2
        elif cmd in ['V','v']:
            y = float(tokens[i+1])
            new_point = (current[0],y)
            lines.append((current,new_point))
            current = new_point
            i += 2
        elif cmd in ['Z','z']:
            if start and current != start:
                lines.append((current,start))
            i += 1
        else:
            i += 1
    return lines

def parse_svg(svg_path, canvas_width=1000, canvas_height=1000):
    tree = ET.parse(svg_path)
    root = tree.getroot()

    rects = []
    lines = []
    polylines = []

    for elem in root.iter():
        tag = elem.tag.split('}')[-1]  # handle namespaces
        if tag == 'rect':
            x = float(elem.attrib.get('x', 0))
            y = float(elem.attrib.get('y', 0))
            w = float(elem.attrib.get('width', 0))
            h = float(elem.attrib.get('height', 0))
            rects.append((x, y, w, h))
        elif tag == 'line':
            x1 = float(elem.attrib.get('x1', 0))
            y1 = float(elem.attrib.get('y1', 0))
            x2 = float(elem.attrib.get('x2', 0))
            y2 = float(elem.attrib.get('y2', 0))
            lines.append(((x1, y1), (x2, y2)))
        elif tag == 'path':
            d = elem.attrib.get('d','')
            path_lines = parse_path(d)
            if path_lines:
                lines.extend(path_lines)   # treat path segments as lines
        elif tag == 'polyline':
            points = elem.attrib.get('points', '').strip().split()
            pts = []
            for p in points:
                if ',' in p and p.strip() != '':
                    try:
                        x, y = map(float, p.split(','))
                        pts.append((x, y))
                    except ValueError:
                        # skip malformed points safely
                        continue
            if pts:  # only append if we got valid points
                polylines.append(pts)


    return rects, lines, polylines

def compute_features(rects, lines, polylines, canvas_width=1000, canvas_height=1000):
    features = {}

    # --- Rectangles ---
    areas = [w*h for (_,_,w,h) in rects]
    features['RectCoverage'] = sum(areas) / (canvas_width*canvas_height) if rects else 0
    features['AvgRectArea'] = sum(areas)/len(areas) if rects else 0
    features['RectStDev'] = math.sqrt(sum((a-features['AvgRectArea'])**2 for a in areas)/len(areas)) if rects else 0
    features['RectDistribution'] = len(set([x for (x,_,_,_) in rects]))/canvas_width if rects else 0
    
    # --- RectOrth: measure grid alignment ---
    if rects:
        xs = [x for (x,_,_,_) in rects]
        ys = [y for (_,y,_,_) in rects]
        # Count how many share coordinates
        aligned_x = sum(1 for i in range(len(xs)) for j in range(i+1,len(xs)) if xs[i]==xs[j])
        aligned_y = sum(1 for i in range(len(ys)) for j in range(i+1,len(ys)) if ys[i]==ys[j])
        total_pairs = len(rects)*(len(rects)-1)/2
        features['RectOrth'] = (aligned_x + aligned_y)/total_pairs if total_pairs>0 else 0
    else:
        features['RectOrth'] = 0.0

    features['RectOrth2'] = 0  # still placeholder
    features['rectangles'] = len(rects)

    # --- Lines ---
    lengths = [math.dist(p1,p2) for p1,p2 in lines]
    features['AvgLineLength'] = sum(lengths)/len(lengths) if lengths else 0
    features['LongestLine'] = max(lengths) if lengths else 0
    features['ShortestLine'] = min(lengths) if lengths else 0
    features['LineLengthStDev'] = math.sqrt(sum((l-features['AvgLineLength'])**2 for l in lengths)/len(lengths)) if lengths else 0
    features['AvgLineAngle'] = sum(math.degrees(math.atan2(p2[1]-p1[1], p2[0]-p1[0])) for p1,p2 in lines)/len(lines) if lines else 0
    features['OrthLinesRatio'] = sum(1 for (p1,p2) in lines if p1[0]==p2[0] or p1[1]==p2[1]) / len(lines) if lines else 0
    features['lines'] = len(lines)

    # --- Line crossings ---
    def intersect(a,b,c,d):
        def ccw(p,q,r): return (r[1]-p[1])*(q[0]-p[0]) > (q[1]-p[1])*(r[0]-p[0])
        return ccw(a,c,d) != ccw(b,c,d) and ccw(a,b,c) != ccw(a,b,d)
    crossings = 0
    for i in range(len(lines)):
        for j in range(i+1,len(lines)):
            if intersect(lines[i][0], lines[i][1], lines[j][0], lines[j][1]):
                crossings += 1
    features['LineCrossings'] = crossings
    features['AvgCrossingAngle'] = 0  # still placeholder

    # --- Polyline bends ---
    bends = []
    for pts in polylines:
        bends.append(len(pts)-2 if len(pts)>2 else 0)
    features['AvgLineBends'] = sum(bends)/len(bends) if bends else 0

    # --- Distances between rectangles ---
    def rect_center(r): return (r[0]+r[2]/2, r[1]+r[3]/2)
    centers = [rect_center(r) for r in rects]
    dists = []
    for i in range(len(centers)):
        for j in range(i+1,len(centers)):
            dists.append(math.dist(centers[i], centers[j]))
    features['AvgShortestDistance'] = sum(dists)/len(dists) if dists else 0

    # --- Aspect ratio ---
    features['AspectRatio'] = canvas_width/canvas_height

    return features



In [2]:
import os
import pandas as pd

def process_folder(folder_name, label):
    data = []
    for file in os.listdir(folder_name):
        if file.endswith(".svg"):
            rects, lines, polylines = parse_svg(os.path.join(folder_name, file))
            feats = compute_features(rects, lines, polylines)
            feats["diagram_name"] = file
            feats["source"] = label
            data.append(feats)
    return pd.DataFrame(data)

# Process all folders
df_ground = process_folder("../../Graph Generation/Ground_truth/SVG", "GroundTruth")
df_llm_a = process_folder("../../Graph Generation/generated_svgs/claude_4_5", "claude_4_5")
df_llm_b = process_folder("../../Graph Generation/generated_svgs/gpt4o", "gpt4o")
df_llm_c = process_folder("../../Graph Generation/generated_svgs/gpt5", "gpt5")

# Combine
df_all = pd.concat([df_ground, df_llm_a, df_llm_b, df_llm_c], ignore_index=True)

# Save
df_all.to_csv("all_features.csv", index=False)